# Main: SciPy parameter identification loop (issue #3)

**Partially runnable now, fully runnable once `surroptim.inverse` stubs are filled in.**
`SensorLocator` and `compute_mse` raise `NotImplementedError` until issue #3's code is
written (`src/surroptim/inverse/sensors.py`, `objective.py`). What this notebook shows,
correctly and ready to use as-is: the `forward_model(theta)` wiring, the
synthetic-data self-test, and the bounded `scipy.optimize.minimize` call.

Built on the **linear** model on purpose (this branch does not depend on issue #2's
non-linear radiation work, which lives on its own branch) -- see the cost warning at
the bottom for what changes once the two are merged together.


In [ ]:
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import mesh, fem
import dolfinx.fem.petsc
from scipy.optimize import minimize

from surroptim.inverse import SensorLocator, compute_mse


## Geometry, spaces, BCs -- unchanged, Constants instead of floats

Same mesh as `notebooks/guidance/constants_and_sensors.ipynb`. `SensorLocator` is built
**once**, right after the mesh, and reused for every forward solve below.


In [ ]:
width, thickness = 1.0, 0.5
nx, ny = 60, 20

domain = mesh.create_rectangle(
    MPI.COMM_WORLD,
    [np.array([0.0, 0.0]), np.array([width, thickness])],
    [nx, ny],
    cell_type=mesh.CellType.triangle,
)
V = fem.functionspace(domain, ("CG", 1))

def boundary_bottom(x):
    return np.isclose(x[1], 0.0)

dofs_bottom = fem.locate_dofs_geometrical(V, boundary_bottom)
bcs = [fem.dirichletbc(PETSc.ScalarType(0.0), dofs_bottom, V)]

r_weight = fem.Function(V)
r_weight.interpolate(lambda x: np.maximum(np.abs(x[0]), 1e-14))

uh = fem.Function(V)
u_n = fem.Function(V)
du = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

def grad_cyl(w):
    return ufl.as_vector([ufl.Dx(w, 0), ufl.Dx(w, 1)])

dt = 5.0e-3
n_steps = 20

thermal_capacity = fem.Constant(domain, PETSc.ScalarType(1.0))
diffusion_coeff = fem.Constant(domain, PETSc.ScalarType(1.0))
advection_coeff = fem.Constant(domain, PETSc.ScalarType(1.0))   # theta[0]: "h"
power_const = fem.Constant(domain, PETSc.ScalarType(1.0))       # theta[1]: source "power"
# (extend with wr, wz, ... the same way once you're comfortable with two parameters)

a_ufl = (
    thermal_capacity * (1.0 / dt) * du * v * r_weight * ufl.dx
    + diffusion_coeff * ufl.dot(grad_cyl(du), grad_cyl(v)) * r_weight * ufl.dx
    + advection_coeff * du * v * r_weight * ufl.ds
)
f_source = fem.Function(V)
f_source.interpolate(lambda x: np.exp(-0.5 * ((x[0] - 0.0) / 0.25) ** 2 - 0.5 * ((x[1] - thickness) / 0.05) ** 2))
L_ufl = (
    thermal_capacity * (1.0 / dt) * u_n * v * r_weight * ufl.dx
    + power_const * f_source * v * r_weight * ufl.dx
)

a = fem.form(a_ufl)
L = fem.form(L_ufl)

sensors_rz = np.array([[0.1, 0.25], [0.5, 0.25], [0.9, 0.25]])
sensor_locator = SensorLocator(domain, sensors_rz)  # built ONCE


## `forward_model(theta)` -- the thing `compute_mse` treats as a black box

This is the only function that touches dolfinx directly. Everything downstream
(`compute_mse`, `scipy.optimize.minimize`) only ever sees plain numpy arrays.


In [ ]:
PARAM_NAMES = ["h", "power"]

def forward_model(theta: dict) -> np.ndarray:
    """Run the forward FE model for given trial parameters, return sensor values.

    Args:
        theta: dict with keys PARAM_NAMES, e.g. {"h": 2.0, "power": 1.3}.

    Returns:
        (3,) array: simulated temperature at the 3 sensors after n_steps.
    """
    advection_coeff.value = theta["h"]
    power_const.value = theta["power"]

    u_n.x.array[:] = 0.0
    uh.x.array[:] = 0.0

    A = dolfinx.fem.petsc.assemble_matrix(a, bcs=bcs)
    A.assemble()
    ksp = PETSc.KSP().create(domain.comm)
    ksp.setOperators(A)
    ksp.setType("preonly")
    ksp.getPC().setType("lu")

    for _ in range(n_steps):
        b = dolfinx.fem.petsc.assemble_vector(L)
        dolfinx.fem.petsc.apply_lifting(b, [a], [bcs])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
        dolfinx.fem.petsc.set_bc(b, bcs)
        ksp.solve(b, uh.x.petsc_vec)
        uh.x.scatter_forward()
        u_n.x.array[:] = uh.x.array

    return sensor_locator.eval(uh)


## Test with synthetic data FIRST (mandatory, per issue #3's mentoring notes)

Generate "measured" data with known parameters, add noise, and try to recover them.
This proves the optimisation loop itself works, independently of any real experimental
measurement error or a wrong physical model.


In [ ]:
rng = np.random.default_rng(seed=0)

theta_truth = {"h": 3.0, "power": 1.5}
measured_clean = forward_model(theta_truth)
noise_level = 0.02 * np.abs(measured_clean).max()
measured = measured_clean + rng.normal(scale=noise_level, size=measured_clean.shape)

print("theta_truth:", theta_truth)
print("synthetic 'measured' data:", measured)


## SciPy coupling -- bounded, because thermal parameters cannot be negative

`bounds=[(0, None), (0, None)]` stops L-BFGS-B from ever asking `forward_model` to
solve the physics with a negative `h` or negative `power` -- do not skip this, per
issue #3's explicit warning.


In [ ]:
def objective(params):
    return compute_mse(
        params,
        param_names=PARAM_NAMES,
        forward_model=forward_model,
        measured=measured,
    )

theta0 = np.array([1.0, 1.0])  # deliberately far from theta_truth = (3.0, 1.5)
bounds = [(0, None), (0, None)]

result = minimize(
    objective,
    theta0,
    method="L-BFGS-B",
    bounds=bounds,
    options={"maxiter": 50},
)

print("converged:", result.success, "-", result.message)
print("recovered:", dict(zip(PARAM_NAMES, result.x)))
print("truth was:", theta_truth)


## Cost warning (issue #3's mentoring note, worth re-reading before scaling this up)

`L-BFGS-B` estimates gradients by finite differences here (no adjoint solver): for
`N=2` parameters that is up to `N+1 = 3` forward PDE solves **per iteration**, not per
optimisation run. With `n_steps=20` linear solves per forward call, one `minimize()`
run above already costs `iterations * 3 * 20` linear solves.

Once this is combined with issue #2's non-linear radiation model, each of those 20
"solves" becomes a full Newton/SNES iteration (several linear solves each) instead of
one direct LU solve -- the N+1 finite-difference cost multiplies directly on top of
that. Before scaling to more than 2-3 parameters, or before switching to the
non-linear model, re-measure how long a single `forward_model` call takes and multiply
by `(N+1) * expected_iterations` -- don't find out the hard way mid-optimisation.
